# CKA Circuit Comparison (IID vs Non-IID)

Use **Centered Kernel Alignment (CKA)** to compare how models and circuits trained under different partitions (e.g., IID vs Non-IID) encode classes.

**Note:** This notebook does NOT train models from scratch. It assumes you have already trained your models and discovered circuits using `FedMI`. You will upload your experiment artifacts (`config.json`, `checkpoints`, `circuits`) to run the comparison.

---

## 1 · Setup environment

In [ ]:
import os
REPO_URL = "https://github.com/ha405/FedMI.git"
BRANCH = "cvpr"

if not os.path.isdir("FedMI"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    !cd FedMI && git pull origin {BRANCH}

os.chdir("FedMI")

from fedmi.env import setup, patch_config, print_info
setup()
print_info()

## 2 · Instructions to Upload Your Experiments

You can now simply upload your `.pt` checkpoint files, your `config.json` files, and your `all_circuits.json` files directly to this notebook environment (e.g. into the `/content` folder in Colab).

Run the cell below to declare the specific paths to the files you uploaded.

In [ ]:
# Set these paths to where you uploaded your files

# File Paths for Model A (e.g. IID)
ckpt_a = "model_a.pt"
cfg_a = "config_a.json"
circ_a = "circuits_a.json"

# File Paths for Model B (e.g. Non-IID)
ckpt_b = "model_b.pt"
cfg_b = "config_b.json"
circ_b = "circuits_b.json"

print("Ensure you have uploaded these 6 files, or change the paths above to match your uploads.")

## 3 · CKA Comparison: Pre-head Latents

Compare the full-model representations (before the classification head) between the two experiments. Do they encode the inputs securely or in visually different latent spaces?

In [ ]:
from fedmi.playground import CKACompareExperiment

class Args:
    pass

args_prehead = Args()
args_prehead.ckpt_a = ckpt_a
args_prehead.cfg_a = cfg_a
args_prehead.circ_a = None # No circuits needed for prehead
args_prehead.exp_a = None

args_prehead.ckpt_b = ckpt_b
args_prehead.cfg_b = cfg_b
args_prehead.circ_b = None
args_prehead.exp_b = None
args_prehead.client_a = 0
args_prehead.client_b = 0
args_prehead.mode = "prehead"
args_prehead.round = None
args_prehead.max_samples = 2000
args_prehead.output = None  # Heatmap skipped for single scalar

cka_exp_prehead = CKACompareExperiment(args_prehead)
cka_exp_prehead.run()

## 4 · CKA Comparison: Cross-Experiment Circuit Matching

Compare the activated circuit for matching classes across the two experiments. A low score implies the networks built structurally or functionally isolated pathways for the same class.

In [ ]:
args_circuit = Args()
args_circuit.ckpt_a = ckpt_a
args_circuit.cfg_a = cfg_a
args_circuit.circ_a = circ_a
args_circuit.exp_a = None

args_circuit.ckpt_b = ckpt_b
args_circuit.cfg_b = cfg_b
args_circuit.circ_b = circ_b
args_circuit.exp_b = None
args_circuit.client_a = 0    # First client from exp_a
args_circuit.client_b = 0    # First client from exp_b
args_circuit.mode = "circuit"
args_circuit.source = "local"
args_circuit.round = None
args_circuit.round_key = "last"
args_circuit.classes = None  # Compute all overlapping classes automatically
args_circuit.layer = None    # Default: last conv/linear before head
args_circuit.max_samples = 2000
args_circuit.output = "cka_heatmap.png"

cka_exp_circuit = CKACompareExperiment(args_circuit)
cka_exp_circuit.run()

## 5 · View Heatmap

In [ ]:
from IPython.display import Image, display
if os.path.exists("cka_heatmap.png"):
    display(Image(filename="cka_heatmap.png"))
else:
    print("Heatmap not generated. Check if there were any overlapping classes.")